In [ ]:
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
import re
from datetime import date
import requests
import ast
import xlwt
from xlwt.Workbook import *
from pandas import ExcelWriter
import xlsxwriter
import time

# Plotting
import matplotlib.pyplot as plt
import plotly
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio

In [ ]:
# Define user
user = os.getlogin()
path_users = os.path.join('C:\\Users', user)

## Set file paths

# SharePoint
path_sp   = os.path.join(path_users, 'Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents')
path_agol = os.path.join(path_sp, 'Process Revamp', 'Task 8. Reproduce Progress Report indicators', 'Indicator Data', 'Census Data')
path_main = os.path.join(path_sp, 'Data')

# Git
path_git  = os.path.join(path_users, 'Documents', 'Projects', 'Regional-Monitoring', 'Indicator_Gen')
path_config0 = os.path.join(path_git, 'config')
path_config  = os.path.join(path_git, 'Python Code', 'Census', 'aa_config')

In [ ]:
## User defined functions
exec(open(os.path.join(path_config0, 'Functions.py')).read())

## Set API key
# Obtain API Key from the following source 
# https://api.census.gov/data/key_signup.html
# Copy retrieved API key to .txt file for safe keeping
file_api = open(os.path.join(path_config, 'api_key.txt'))
api_key = file_api.read()
file_api.close()

In [ ]:
indicator_name = 'Accessibility_2'
estimate = 'ACS1'


In [ ]:
df_acs = pd.read_csv(os.path.join(path_agol, indicator_name, 'Inter', indicator_name + '_PUMA_' + estimate + '.csv')
                                    , dtype = {'State FIPS': object, 'PUMA': object, 'SERIALNO': object})

df_cpi = pd.read_excel(os.path.join(path_git, 'config', 'California CPI.xlsx'), sheet_name = 'Inflation Adjustment Factors')

In [ ]:
df_acs.head()

In [ ]:
df_cpi = df_cpi[['Year', 'IAF_2022']].rename(columns = {'Year':'year'})
df_cpi.head()

In [ ]:
df_acs = df_acs.merge(df_cpi, on = 'year', how = 'left')
df_acs['PINCP_IAF'] = df_acs['PINCP']*df_acs['ADJINC']*df_acs['IAF_2022']
df_acs.head()

In [ ]:
# Renters vs Owners
conditions = [
                ( (df_acs['PINCP'] == 0) ),
                ( (df_acs['PINCP']  < 0) ),
                ( (df_acs['PINCP_IAF'] > 0) & (df_acs['PINCP_IAF'] < 40000) ),
                ( (df_acs['PINCP_IAF'] >=  40000) & (df_acs['PINCP_IAF'] <  80000) ),
                ( (df_acs['PINCP_IAF'] >=  80000) & (df_acs['PINCP_IAF'] < 120000) ),
                ( (df_acs['PINCP_IAF'] >= 120000) )
            ]

choices = ['None', 'Loss of income', '0 to 40k', '40k to 80k', '80k to 120k', '120k or higher']
df_acs["income_bracket"] = np.select(conditions, choices)
df_acs.head()

In [ ]:
df_fips1 = pd.read_excel(os.path.join(path_config0, 'Area Codes.xlsx'), sheet_name = 'CountyFIPS' 
                                , dtype = {'State FIPS': object, 'County FIPS': object})
df_fips2 = pd.read_excel(os.path.join(path_config0, 'Area Codes.xlsx'), sheet_name = 'PUMAcodes' 
                                , dtype = {'STATEFP': object, 'COUNTYFP': object})

df_fips1 = df_fips1[df_fips1['State FIPS'].isin(['06'])]
df_fips2 = df_fips2[df_fips2['STATEFP'   ].isin(['06'])]

df_fips2['PUMA5CE'] = df_fips2['PUMA5CE'].astype(str).apply('{:0>5}'.format)
df_fips2 = df_fips2[['STATEFP', 'COUNTYFP', 'PUMA5CE', 'Years']].drop_duplicates()
df_fips2 = df_fips2.rename(columns = {'STATEFP':'State FIPS', 'COUNTYFP':'County FIPS', 'PUMA5CE':'PUMA'})

df_acs['PUMA'] = df_acs['PUMA'].astype(str).apply('{:0>5}'.format)

df_fips1 = df_fips1[['State FIPS', 'County FIPS', 'County Name', 'MPO']]# , 'MSA']]

df_acs2 = df_acs[df_acs['year'].isin(sequence(2022, 2031, 1))].merge(df_fips2[df_fips2['Years'] == '2022-2031'], on = ['State FIPS', 'PUMA'])
df_acs1 = df_acs[df_acs['year'].isin(sequence(2012, 2021, 1))].merge(df_fips2[df_fips2['Years'] == '2012-2021'], on = ['State FIPS', 'PUMA'])
df_acs = pd.concat([df_acs1, df_acs2])

df_acs = df_acs.merge(df_fips1, on = ['State FIPS', 'County FIPS'])

df_acs.head(3)

In [ ]:
# df_acs2 = df_acs.groupby(['State FIPS', 'PUMA', 'PUMA NAME', 'year', 'JWTRNS', 'income_bracket'], as_index = False)['PWGTP'].sum()
df_acs2 = df_acs.groupby(['State FIPS', 'County Name', 'year', 'income_bracket', 'JWTRNS'], as_index = False)['PWGTP'].sum()

df_acs2['Percentage'] = 100*df_acs2['PWGTP'] / df_acs2.groupby(['County Name', 'year', 'income_bracket'])['PWGTP'].transform('sum')

df_acs2

In [ ]:
# Create "Categorical" race/ethnicity field for sorting
# Sort by geography, variable mapping, and race/ethnicity
# sort and then remove categorical field
df_acs2['Income_sort'] = pd.Categorical(df_acs2['income_bracket'], ['None'
                                                                     , 'Loss of income'
                                                                     , '0 to 40k'
                                                                     , '40k to 80k'
                                                                     , '80k to 120k'
                                                                     , '120k or higher'
                                                                    ])

df_acs2['JWTRNS_sort'] = pd.Categorical(df_acs2['JWTRNS'], [
       'Car, truck, or van'
       , 'Public transportation (bus, subway, or rail)'
       , 'Bicycle'
       , 'Walked'
       , 'Worked from home'
       , 'Other method'
])

df_acs2 = df_acs2.sort_values(by = ['County Name', 'year', 'Income_sort', 'JWTRNS_sort'], ascending = [True, False, True, True]) 
df_acs2 = df_acs2.drop(['JWTRNS_sort', 'Income_sort'], axis = 1)
df_acs2

In [ ]:
print('Columns: ' + str(list(df_acs2.columns)))

In [ ]:
df_plot = df_acs2[df_acs2['income_bracket'].isin(['0 to 40k', '40k to 80k', '80k to 120k', '120k or higher'])]
df_plot = df_plot[df_plot['County Name'] == 'Placer']

# Plotting setup
by_race = False
race_ethnicity = 'race_ethnicity'
by_vars = False
variable = 'JWTRNS'
x = 'year'
y = 'Percentage'
color = 'JWTRNS'
line_dash = 'income_bracket'
markers = True
plot_title = 'Commute Modes by Income Level'
plot_name = 'Median Household Income by Peer MSA'
export = False

In [ ]:

def plot_lines(
    df=df_plot
     , by_vars=by_vars, by_race=by_race, race_ethnicity=race_ethnicity, variable=variable
     , x=x, y=y
     , color=color, line_dash=line_dash, markers=markers
     , plot_title=plot_title, plot_name=plot_name
     , export=export
):
    

    if by_vars == True:
        vars = unique(df[variable].values)
        for var in vars:
            df2 = df[df[variable] == var]
            fig = px.line(df2, x = x, y = y, color = color, line_dash = line_dash, markers = markers)
            fig.update_layout(title = plot_title + ' - ' + str(var))
            if export == True:
                fig.write_html(os.path.join(path_plots, ''.join([indicator_name + '_', plot_name + '_', var + '_', 'line.html'])))
                
            fig.update_layout(autosize=False, width=1050, height=450)
            
    else:
        fig = px.line(df, x = x, y = y, color = color, line_dash = line_dash, markers = markers)
        fig.update_layout(title = plot_title)
        if export == True:
            fig.write_html(os.path.join(path_plots, ''.join([indicator_name + '_', plot_name + '_', 'line.html'])))

        fig.update_layout(autosize=False, width=1050, height=450)

    return fig.show()
        
plot_lines()